In [ ]:
import os

In [ ]:
import joblib
import matplotlib.pyplot as plt
import mlflow
import pandas as pd
import seaborn as sns

In [ ]:
from dotenv import load_dotenv
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

load_dotenv()

In [ ]:
path_train = "../data/raw/full.csv"

try:
    data = pd.read_csv(path_train)
    print(f"Успешно загружен датасет: {data.shape}")
except:
    print(f"Файл не найден: {path_train}")

In [ ]:
data.head(5)

In [ ]:
data.shape

In [ ]:
sns.countplot(x="Survived", hue="Sex", data=data)

In [ ]:
sns.countplot(x="Survived", hue="Pclass", data=data)

In [ ]:
data["Age"].plot.hist()

In [ ]:
data.isnull().sum()

In [ ]:
data.head(5)

In [ ]:
data = data.drop(
    [
        "Name_wiki",
        "Name",
        "Destination",
        "Hometown",
        "Ticket",
        "PassengerId",
        "WikiId",
        "Boarded",
        "Body",
        "Age_wiki",
        "Lifeboat",
        "Cabin",
    ],
    axis=1,
)

In [ ]:
embarked = pd.get_dummies(
    data["Embarked"],
).astype(float)

data = pd.concat([data, embarked], axis=1)
data = data.drop("Embarked", axis=1)

In [ ]:
data = data.dropna()
data = data.drop_duplicates()

In [ ]:
print((data.isna().sum() / len(data)) * 100)

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(data.isnull(), cbar=True, yticklabels=False, cmap="viridis")
plt.title("Карта пропусков в данных")
plt.show()

In [ ]:
data.head(5)

In [ ]:
print(data.select_dtypes(include=["object"]).columns.tolist())

In [ ]:
x = data.drop("Survived", axis=1)
y = data["Survived"]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=42, stratify=y
)

In [ ]:
try:
    model_load = joblib.load("./model.pkl")
    print("Модель загружена из model.pkl")
except FileNotFoundError:
    print("Файл model.pkl не найден, модель будет обучена заново")

In [ ]:
scaler = StandardScaler()
numeric_features = ["Fare", "Class", "SibSp", "Age"]
categorical_features = ["Sex"]
numeric_transform = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", scaler),
    ]
)

categorical_transform = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    [
        ("num", numeric_transform, numeric_features),
        ("cat", categorical_transform, categorical_features),
    ]
)

In [ ]:
try:
    if mlflow.active_run():
        mlflow.end_run()
except Exception as e:
    print(f"Предупреждение: не удалось завершить предыдущий запуск: {e}")

In [ ]:
if "__file__" in locals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.path.abspath("")

DB_DIR = os.path.join(current_dir, "./db_models")
if not os.path.exists(DB_DIR):
    os.makedirs(DB_DIR)

DB_PATH = os.path.join(DB_DIR, "./mlflow.db")
db_uri = f"sqlite:///{DB_PATH}"

mlflow.set_tracking_uri(db_uri)


mlflow.set_experiment("Titanic Project")
mlflow.sklearn.autolog()
mlflow.enable_system_metrics_logging()
with mlflow.start_run(run_name="Hyperparameter"):
    model = LogisticRegression(max_iter=1000, random_state=43)
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])

    param_grid = [
        {
            "model__l1_ratio": [1.0],
            "model__C": [0.1, 1, 10],
            "model__solver": ["liblinear", "saga"],
        },
        {
            "model__l1_ratio": [0.0],
            "model__C": [0.1, 1, 10],
            "model__solver": ["lbfgs", "liblinear", "saga"],
        },
        {
            "model__l1_ratio": [0.1, 0.5, 0.9],
            "model__C": [0.1, 1, 10],
            "model__solver": ["saga"],
        },
    ]

    grid_model = GridSearchCV(pipeline, param_grid=param_grid, cv=5, scoring="f1_macro")
    grid_model.fit(x_train, y_train)

    predict = grid_model.predict(x_test)
    report = classification_report(y_test, predict, output_dict=True)

    mlflow.log_params(grid_model.best_params_)
    mlflow.log_metric("f1_score", grid_model.best_score_)
    mlflow.log_metric("test_accuracy", report["accuracy"])
    mlflow.log_metric("test_precision", report["macro avg"]["precision"])
    mlflow.log_metric("test_recall", report["macro avg"]["recall"])

    report_file = "classification_report.txt"
    with open(report_file, "w") as f:
        f.write(classification_report(y_test, predict))
    mlflow.log_artifact(report_file)

    mlflow.sklearn.log_model(
        sk_model=grid_model.best_estimator_,
        artifact_path="best_model",
        registered_model_name="LogisticRegressionBest",
    )

print(f"Успешно! Данные в базе: {DB_PATH}")

In [ ]:
print("Текущий URI трекера:", mlflow.get_tracking_uri())

In [ ]:
joblib.dump(grid_model.best_estimator_, "model.pkl")

In [ ]:
cm = confusion_matrix(y_test, predict)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Матрица ошибок")
plt.ylabel("Истинные метки")
plt.ylabel("Предсказание метрики")
plt.savefig("confusion_matrix.png")
mlflow.log_artifact("confusion_matrix.png")
plt.show()